# Pelican file events: live EarthScope GNSS data

This notebook subscribes to a Pelican namespace, and plots each new
object as it appears. The data is EarthScope GNSS displacement:
east, north and up, one file per minute.

Nothing is downloaded. Each object is read straight from the
federation into memory, and the NDP Endpoint is not in the data path.

Everything here goes through `ndp-ep`. There is no STOMP, no
WebSocket handling and no `asyncio` to write.

In [ ]:
%pip install "ndp-ep[pelican]"

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

## Configuration

**Choose your own `CLIENT_ID`.** It identifies your subscriber. Two
clients sharing an id compete for the same events, so reusing someone
else's will take events away from them.

`EVENT_SOURCE` is the namespace to watch. The credentials are checked
by the event server against its own store; they are unrelated to the
Endpoint token, and will be replaced by an access token later.

In [ ]:
CLIENT_ID = "my-client"  # <- change this
EVENT_SOURCE = "osdf/vdc/public/pelican_protocol"

ENDPOINT_URL = "http://155.101.6.191:8003"
EVENT_USERNAME = "your-username"
EVENT_PASSWORD = "your-password"

## Connect and subscribe

One client object covers both halves: the subscription that tells you
an object appeared, and the reads that fetch it.

In [ ]:
from ndp_ep import APIClient

client = APIClient(base_url=ENDPOINT_URL)

subscription = client.subscribe_pelican(
    EVENT_SOURCE,
    client_id=CLIENT_ID,
    username=EVENT_USERNAME,
    password=EVENT_PASSWORD,
)

subscription.wait_until_connected(timeout=30)
subscription.status

## Plot each file as it arrives

`subscription.events(timeout=...)` blocks until the next event and
yields it once. Redeliveries are suppressed against a record on disk,
so no bookkeeping is needed here and restarting the notebook will not
reprocess what it already plotted.

The publisher writes roughly one file per minute, but a newly
connected subscriber usually sees nothing for the first few minutes:
measured runs waited between two and four. The server then delivers
what it has queued in a burst, so those first events arrive seconds
apart rather than a minute apart. Expect the cell below to take
several minutes for three events, most of it before the first one,
and do not read the initial silence as a failure.

`EVENT_TIMEOUT` bounds the wait for each individual event, so the cell
always ends instead of hanging if the publisher goes quiet. It has to
be comfortably larger than that startup delay. Raise `MAX_EVENTS` to
keep the cell running longer.

A new subscriber does not replay the namespace's history, but the
first burst can include objects written shortly before it connected.

In [ ]:
%pip install plotly anywidget pandas

In [ ]:
import io

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

MAX_POINTS = 600
MAX_EVENTS = 3
EVENT_TIMEOUT = 420  # seconds; well above the startup delay

all_data = pd.DataFrame()


def create_figure(title, color):
    fig = go.FigureWidget()
    fig.add_scatter(mode="lines", line=dict(color=color, width=2),
                    name=title)
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=250,
        margin=dict(l=50, r=30, t=40, b=40),
        xaxis_title="Time",
        yaxis_title=title,
    )
    return fig


east_fig = create_figure("East", "royalblue")
north_fig = create_figure("North", "green")
up_fig = create_figure("Up", "firebrick")

display(east_fig)
display(north_fig)
display(up_fig)

for received, event in enumerate(
    subscription.events(timeout=EVENT_TIMEOUT), start=1
):

    print(f"Received {event.name}")

    raw = client.pelican_read(event.url)

    df = pd.read_csv(io.BytesIO(raw))
    df["datetime"] = pd.to_datetime(df["time"], unit="ms", utc=True)

    all_data = pd.concat([all_data, df], ignore_index=True)
    if len(all_data) > MAX_POINTS:
        all_data = all_data.iloc[-MAX_POINTS:].copy()

    x = all_data["datetime"]
    for figure, column in (
        (east_fig, "east"),
        (north_fig, "north"),
        (up_fig, "up"),
    ):
        with figure.batch_update():
            figure.data[0].x = x
            figure.data[0].y = all_data[column]

    if received >= MAX_EVENTS:
        break

## All three components on one figure

In [ ]:
combined = go.FigureWidget()

for column, color in (("east", "royalblue"),
                     ("north", "green"),
                     ("up", "firebrick")):
    combined.add_scatter(
        x=all_data["datetime"],
        y=all_data[column],
        name=column.capitalize(),
        line=dict(color=color, width=2),
    )

combined.update_layout(template="plotly_white", height=350,
                       xaxis_title="Time")
display(combined)

## Browsing without subscribing

The namespace can be listed and read directly, with no subscription
involved. References are accepted in any of the spellings that turn
up in practice: a bare path, `osdf://...`, or `pelican://host/...`.

`pelican_list` returns names in lexicographic order, not chronological
order: `AGMT.CI.LY_.20_c100.csv` sorts before `AGMT.CI.LY_.20_c99.csv`.
The end of the list is therefore not the newest data. Pick a name
explicitly, as below, and use an event's `url` when what you want is
the most recent object.

In [ ]:
objects = client.pelican_list(EVENT_SOURCE)

print(f"{len(objects)} objects")

sample = objects[0]
sample

In [ ]:
raw = client.pelican_read(sample)

print(raw[:200].decode())

## Downloading to disk

`pelican_read` keeps the object in memory. Use `pelican_fetch` when
a file on disk is what you want; an existing target is an error
rather than a silent overwrite.

In [ ]:
path = client.pelican_fetch(sample, "./data/")

print(path, path.stat().st_size, "bytes")

## The raw events

Each event carries the object's name, a reference ready to pass to
`pelican_read`, its size, and the modification time reported by the
server.

Passing a `timeout` bounds the wait, so the cell ends even if the
publisher goes quiet. Keep it well above the few-minute startup delay:
on a subscription that has just connected, a shorter bound returns
nothing and looks like a failure rather than a timeout.

In [ ]:
for received, event in enumerate(
    subscription.events(timeout=EVENT_TIMEOUT), 1
):
    print(event.name, event.size, event.mod_time)
    print("   ", event.url)
    if received >= 2:
        break

## Closing

Closing performs the WebSocket closing handshake, so the event server
does not sit on a half-open connection. Using the subscription as a
context manager (`with client.subscribe_pelican(...) as subscription:`)
does this automatically.

In [ ]:
subscription.close()
subscription.status["metrics"]